# LeetCode #153 — Find Minimum in Rotated Sorted Array: tìm nhị phân điểm gãy

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search

Skeleton trên LeetCode:

```python
class Solution:
    def findMin(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Hiểu mảng **đã sắp xếp bị xoay**: hai đoạn tăng nối nhau, phần tử nhỏ nhất ở **điểm gãy**.
2. Tìm nhị phân bằng cách so sánh `nums[mid]` với `nums[hi]` (phần tử **cuối** khoảng tìm).
3. Hiểu vì sao so với `nums[hi]` an toàn hơn so với `nums[lo]`.
4. Mẫu `while lo < hi` với `hi = mid` / `lo = mid + 1` — tìm vị trí **đầu tiên** thỏa điều kiện.

# 1. Bài toán là gì?

Mảng tăng dần `nums` gồm `n` phần tử **khác nhau**, bị xoay từ `1` đến `n` lần. Ví dụ `[0,1,2,4,5,6,7]` có thể thành:

- `[4,5,6,7,0,1,2]` nếu xoay 4 lần,
- `[0,1,2,4,5,6,7]` nếu xoay 7 lần (trở về như cũ).

Tìm phần tử **nhỏ nhất**, với thời gian `O(log n)`.

**Constraints (theo đề):**

- `n == nums.length`, `1 <= n <= 5000`
- `-5000 <= nums[i] <= 5000`
- Mọi phần tử **khác nhau**.
- `nums` là mảng tăng dần bị xoay từ `1` đến `n` lần.

---

## Ví dụ

```text
[3, 4, 5, 1, 2]           ->  1
[4, 5, 6, 7, 0, 1, 2]     ->  0
[11, 13, 15, 17]          ->  11   (xoay n lần = không xoay)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Áp dụng vào bài này

**Tìm nhị phân dạng "vị trí đầu tiên thỏa điều kiện"**: điều kiện `nums[i] <= nums[-1]` đơn điệu (`False … False True … True`) trên mảng xoay; phần tử nhỏ nhất là vị trí `True` đầu tiên. Khung `while lo < hi`, `hi = mid` / `lo = mid + 1`.

# 3. Nền tảng

In [4]:
from typing import List
import random, time

def rotate(a: List[int], k: int) -> List[int]:
    """Xoay phải k lần (phần tử cuối lên đầu, k lần)."""
    k %= len(a)
    return a[-k:] + a[:-k] if k else a[:]

base = [0, 1, 2, 4, 5, 6, 7]
for k in [0, 1, 4, 7]:
    print(k, rotate(base, k))

0 [0, 1, 2, 4, 5, 6, 7]
1 [7, 0, 1, 2, 4, 5, 6]
4 [4, 5, 6, 7, 0, 1, 2]
7 [0, 1, 2, 4, 5, 6, 7]


## Hình dạng của mảng xoay

```text
   giá trị
     ^      ●
     |    ●            đoạn TRÁI (lớn): mọi phần tử > nums[-1]
     |  ●
     |                    ●
     |                  ●       đoạn PHẢI (nhỏ): mọi phần tử <= nums[-1]
     |                ●  ← min
     +-------------------------> chỉ số
```

Phần tử nhỏ nhất = phần tử **đầu tiên** của đoạn phải = phần tử đầu tiên `<= nums[-1]`.

# 4. Baseline: `min(nums)` — `O(n)`

Đúng, nhưng đề yêu cầu `O(log n)`. Ta dùng nó làm đáp án đối chiếu.

In [5]:
def find_min_linear(nums: List[int]) -> int:
    return min(nums)

print(find_min_linear([3, 4, 5, 1, 2]))

1


# 5. Ý tưởng chính

Điều kiện `P(i) = nums[i] <= nums[-1]` là **đơn điệu**: `False … False True … True`. Ta cần vị trí `True` đầu tiên → tìm nhị phân.

Tương đương và gọn hơn: so `nums[mid]` với `nums[hi]` của khoảng hiện tại:

```text
lo, hi = 0, n - 1
while lo < hi:
    mid = (lo + hi) // 2
    if nums[mid] > nums[hi]:     # mid ở đoạn TRÁI → min ở bên phải mid
        lo = mid + 1
    else:                        # mid ở đoạn PHẢI → min ở mid hoặc bên trái
        hi = mid
return nums[lo]
```

Bất biến: **min luôn nằm trong `[lo, hi]`**. Khi `lo == hi`, đó chính là min.

In [6]:
def find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]

print(find_min([3, 4, 5, 1, 2]), find_min([4, 5, 6, 7, 0, 1, 2]), find_min([11, 13, 15, 17]))

1 0 11


# 6. Mô phỏng

In [7]:
def trace_find_min(nums):
    lo, hi = 0, len(nums) - 1
    step = 0
    while lo < hi:
        mid = (lo + hi) // 2
        step += 1
        if nums[mid] > nums[hi]:
            action = f"nums[mid]={nums[mid]} > nums[hi]={nums[hi]} → lo = {mid + 1}"
            lo = mid + 1
        else:
            action = f"nums[mid]={nums[mid]} <= nums[hi]={nums[hi]} → hi = {mid}"
            hi = mid
        print(f"bước {step}: mid={mid} | {action:<40} | khoảng còn {nums[lo:hi + 1]}")
    print(f"→ min = nums[{lo}] = {nums[lo]}")
    return nums[lo]

_ = trace_find_min([4, 5, 6, 7, 0, 1, 2])

bước 1: mid=3 | nums[mid]=7 > nums[hi]=2 → lo = 4        | khoảng còn [0, 1, 2]
bước 2: mid=5 | nums[mid]=1 <= nums[hi]=2 → hi = 5       | khoảng còn [0, 1]
bước 3: mid=4 | nums[mid]=0 <= nums[hi]=1 → hi = 4       | khoảng còn [0]
→ min = nums[4] = 0


# 7. Phiên bản đúng format LeetCode

In [8]:
class Solution:
    def findMin(self, nums: List[int]) -> int:
        lo, hi = 0, len(nums) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] > nums[hi]:
                lo = mid + 1
            else:
                hi = mid
        return nums[lo]

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.findMin([3, 4, 5, 1, 2]), 1)
check("Ví dụ 2", sol.findMin([4, 5, 6, 7, 0, 1, 2]), 0)
check("Ví dụ 3", sol.findMin([11, 13, 15, 17]), 11)
check("Một phần tử", sol.findMin([5]), 5)
check("Hai phần tử", sol.findMin([2, 1]), 1)

random.seed(153)
total = 0
for n in range(1, 30):
    for _ in range(30):
        base = sorted(random.sample(range(-5000, 5001), n))
        for k in range(n):
            arr = rotate(base, k)
            assert sol.findMin(arr) == min(arr), arr
            total += 1
print(f"{total} mảng xoay (mọi độ xoay): khớp min()")

class Counter(list):
    reads = 0
    def __getitem__(self, i):
        Counter.reads += 1
        return super().__getitem__(i)

base = list(range(5000))
worst = 0
for k in range(0, 5000, 7):
    Counter.reads = 0
    sol.findMin(Counter(rotate(base, k)))
    worst = max(worst, Counter.reads)
print("n = 5000: số lần đọc phần tử nhiều nhất =", worst)

Ví dụ 1          PASS  1
Ví dụ 2          PASS  0
Ví dụ 3          PASS  11
Một phần tử      PASS  5
Hai phần tử      PASS  1
13050 mảng xoay (mọi độ xoay): khớp min()


n = 5000: số lần đọc phần tử nhiều nhất = 27


`⌈log₂ 5000⌉ = 13` vòng, mỗi vòng đọc 2 phần tử + 1 lần cuối → khoảng 27 lần đọc, so với 5000 của `min()`.

# 8. Vì sao đúng? Độ phức tạp?

**Bất biến:** min nằm trong `nums[lo..hi]`. Ban đầu đúng.

- `nums[mid] > nums[hi]`: trong khoảng `[lo, hi]` có điểm gãy nằm **sau** `mid` (nếu `[mid, hi]` tăng dần thì `nums[mid] < nums[hi]`). Mọi `nums[lo..mid]` đều lớn hơn `nums[hi]` (cùng đoạn trái hoặc trước đó) → min ở `[mid+1, hi]`.
- `nums[mid] < nums[hi]` (khác nhau nên không bằng, trừ khi `mid == hi` — không xảy ra vì `mid < hi`): `[mid, hi]` tăng dần → min của nó là `nums[mid]`; min tổng ở `[lo, mid]`.

Mỗi vòng thu hẹp khoảng (vì `mid < hi` và `mid + 1 > lo`) → kết thúc.

```text
Time:  O(log n)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: So với `nums[lo]` thay vì `nums[hi]`

In [10]:
def wrong_compare_lo(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= nums[lo]:                  # SAI: "mid ở đoạn trái" không đúng khi mảng KHÔNG xoay
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]

print("Sai :", wrong_compare_lo([11, 13, 15, 17]), "| Đúng:", sol.findMin([11, 13, 15, 17]))
print("Sai :", wrong_compare_lo([3, 4, 5, 1, 2]), "| Đúng:", sol.findMin([3, 4, 5, 1, 2]))

Sai : 17 | Đúng: 11
Sai : 2 | Đúng: 1


`nums[mid] >= nums[lo]` chỉ nói `[lo, mid]` tăng dần — **không** cho biết min ở bên nào (nếu cả khoảng đã tăng dần, min là `nums[lo]`, nhưng ta lại bỏ nó đi). So với `nums[hi]` không có nhập nhằng này.

## Lỗi 2: `hi = mid - 1` khi `nums[mid] <= nums[hi]`

In [11]:
def wrong_hi_minus(nums):
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1                           # SAI: mid có thể chính là min
    return nums[lo]

print("Sai :", wrong_hi_minus([3, 4, 0, 1, 2]), "| Đúng:", sol.findMin([3, 4, 0, 1, 2]))
print("Ví dụ 2 vẫn tình cờ đúng:", wrong_hi_minus([4, 5, 6, 7, 0, 1, 2]))

Sai : 3 | Đúng: 0
Ví dụ 2 vẫn tình cờ đúng: 0


Với `[3, 4, 0, 1, 2]`: `mid = 2` trỏ đúng vào min `0`, nhưng `hi = mid - 1` loại nó khỏi khoảng tìm. Test chỉ với ví dụ của đề có thể không lộ lỗi này.

## Lỗi 3: `while lo <= hi` với `hi = mid`

Khi `lo == hi`, `mid == hi`, `nums[mid] > nums[hi]` sai → `hi = mid` không đổi → **lặp vô hạn**. Mẫu "tìm vị trí đầu tiên" dùng `lo < hi`.

In [12]:
def wrong_le(nums, limit=100):
    lo, hi, steps = 0, len(nums) - 1, 0
    while lo <= hi and steps < limit:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
        steps += 1
    return steps

print("Số vòng (bị chặn ở 100):", wrong_le([3, 4, 5, 1, 2]))

Số vòng (bị chặn ở 100): 100


# 10. Mẹo nhận diện pattern

- **"`O(log n)` trên mảng gần sắp xếp"** → tìm nhị phân trên một **điều kiện đơn điệu** (không nhất thiết là "giá trị = target").
- **Tìm vị trí đầu tiên thỏa điều kiện** → `while lo < hi`, `hi = mid` / `lo = mid + 1`.
- **Mảng xoay** → so với đầu mút **phải** để biết `mid` thuộc đoạn nào.

Bài liên quan: **#33 Search in Rotated Sorted Array**, **#154 Find Minimum in Rotated Sorted Array II** (Hard — có trùng), **#81 Search in Rotated Sorted Array II**, **#162 Find Peak Element**, **#852 Peak Index in a Mountain Array**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy thuật toán trên `[5, 6, 7, 8, 9, 1, 2, 3, 4]`, ghi `lo`, `hi`, `mid` từng bước.

## Bài 2 — Điền chỗ trống

In [13]:
def exercise_2_find_min(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        # TODO: so nums[mid] với nums[hi]
        pass
    return nums[lo]


# assert exercise_2_find_min([3, 4, 5, 1, 2]) == 1
# assert exercise_2_find_min([4, 5, 6, 7, 0, 1, 2]) == 0
# assert exercise_2_find_min([11, 13, 15, 17]) == 11

## Bài 3 — Biến thể: số lần đã xoay

Trả về `k` — số lần xoay **phải** (modulo `n`) — chính là **chỉ số** của phần tử nhỏ nhất.

In [14]:
def exercise_3_rotation_count(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_3_rotation_count([3, 4, 5, 1, 2]) == 3
# assert exercise_3_rotation_count([11, 13, 15, 17]) == 0
# assert exercise_3_rotation_count([2, 1]) == 1

## Bài 4 — Biến thể: Search in Rotated Sorted Array (LeetCode #33)

Tìm chỉ số của `target` (hoặc `-1`) trong `O(log n)`. Cách đơn giản: tìm chỉ số min `k` (bài 3), rồi tìm nhị phân trên đoạn thích hợp `[0, k-1]` hoặc `[k, n-1]`.

In [15]:
def exercise_4_search(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_4_search([4, 5, 6, 7, 0, 1, 2], 0) == 4
# assert exercise_4_search([4, 5, 6, 7, 0, 1, 2], 3) == -1
# assert exercise_4_search([1], 0) == -1
# assert exercise_4_search([1, 3], 3) == 1

## Bài 5 — Mini challenge: Find Minimum in Rotated Sorted Array II (LeetCode #154, Hard)

Như bài gốc nhưng **có thể trùng**. Khi `nums[mid] == nums[hi]` không biết bên nào → `hi -= 1` (an toàn vì `nums[mid]` vẫn còn giữ giá trị đó). Xấu nhất `O(n)`.

In [16]:
def exercise_5_find_min_dup(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_find_min_dup([1, 3, 5]) == 1
# assert exercise_5_find_min_dup([2, 2, 2, 0, 1]) == 0
# assert exercise_5_find_min_dup([3, 1, 3, 3, 3]) == 1
# assert exercise_5_find_min_dup([3, 3, 3, 1, 3]) == 1

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_find_min` sau khi tự làm.

**Bài 3**
- Mức 1: giống bài 2 nhưng trả `lo` thay vì `nums[lo]`.

**Bài 4**
- Mức 1: `k = chỉ số min`. Nếu `target >= nums[k]` và `target <= nums[-1]` → tìm trong `[k, n-1]`, ngược lại trong `[0, k-1]`.
- Mức 2: dùng `bisect_left` trên khoảng đó và kiểm tra bằng.

**Bài 5**
- Mức 1: thêm nhánh `elif nums[mid] == nums[hi]: hi -= 1`.
- Mức 2: vì sao không bỏ mất min? Nếu `nums[hi]` là min thì `nums[mid]` cũng bằng min và vẫn nằm trong `[lo, hi-1]`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
_ = trace_find_min([5, 6, 7, 8, 9, 1, 2, 3, 4])

bước 1: mid=4 | nums[mid]=9 > nums[hi]=4 → lo = 5        | khoảng còn [1, 2, 3, 4]
bước 2: mid=6 | nums[mid]=2 <= nums[hi]=4 → hi = 6       | khoảng còn [1, 2]
bước 3: mid=5 | nums[mid]=1 <= nums[hi]=2 → hi = 5       | khoảng còn [1]
→ min = nums[5] = 1


## Đáp án Bài 2

In [18]:
def answer_2_find_min(nums: List[int]) -> int:
    return Solution().findMin(nums)

assert answer_2_find_min([3, 4, 5, 1, 2]) == 1
assert answer_2_find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert answer_2_find_min([11, 13, 15, 17]) == 11
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [19]:
def answer_3_rotation_count(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert answer_3_rotation_count([3, 4, 5, 1, 2]) == 3
assert answer_3_rotation_count([11, 13, 15, 17]) == 0
assert answer_3_rotation_count([2, 1]) == 1
for n in range(1, 15):
    base = list(range(n))
    for k in range(n):
        assert answer_3_rotation_count(rotate(base, k)) == k
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [20]:
from bisect import bisect_left

def answer_4_search(nums: List[int], target: int) -> int:
    n = len(nums)
    k = answer_3_rotation_count(nums)
    if nums[k] <= target <= nums[-1]:
        lo, hi = k, n
    else:
        lo, hi = 0, k
    i = bisect_left(nums, target, lo, hi)
    return i if i < hi and nums[i] == target else -1

assert answer_4_search([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert answer_4_search([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert answer_4_search([1], 0) == -1
assert answer_4_search([1, 3], 3) == 1
for _ in range(3000):
    n = random.randint(1, 12)
    arr = rotate(sorted(random.sample(range(-20, 21), n)), random.randint(0, n - 1))
    t = random.randint(-22, 22)
    assert answer_4_search(arr, t) == (arr.index(t) if t in arr else -1), (arr, t)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [21]:
def answer_5_find_min_dup(nums: List[int]) -> int:
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        elif nums[mid] < nums[hi]:
            hi = mid
        else:
            hi -= 1
    return nums[lo]

assert answer_5_find_min_dup([1, 3, 5]) == 1
assert answer_5_find_min_dup([2, 2, 2, 0, 1]) == 0
assert answer_5_find_min_dup([3, 1, 3, 3, 3]) == 1
assert answer_5_find_min_dup([3, 3, 3, 1, 3]) == 1
for _ in range(5000):
    n = random.randint(1, 10)
    base = sorted(random.randint(0, 3) for _ in range(n))
    arr = rotate(base, random.randint(0, n - 1))
    assert answer_5_find_min_dup(arr) == min(arr), arr
print("Bài 5 PASS")
print("Bài gốc gặp trùng có thể sai: find_min([3, 3, 3, 1, 3]) =", find_min([3, 3, 3, 1, 3]))

Bài 5 PASS
Bài gốc gặp trùng có thể sai: find_min([3, 3, 3, 1, 3]) = 3


# 14. Tổng kết

> Mảng xoay = hai đoạn tăng; min là phần tử đầu tiên của đoạn phải. Tìm nhị phân với `while lo < hi`: `nums[mid] > nums[hi]` → `lo = mid + 1`, ngược lại `hi = mid`. Trả `nums[lo]`. So với **`nums[hi]`**, không phải `nums[lo]`. `O(log n)`.

---

## Checklist trước khi nộp

- [ ] So `nums[mid]` với `nums[hi]`?
- [ ] `hi = mid` (không `mid - 1`)?
- [ ] `while lo < hi` (không `<=`)?
- [ ] Đúng cả khi mảng **không** xoay và khi `n = 1`?